# XRPL Payment Fraud Detection

This notebook uses the labelled XRPL-style payment transactions in `train/` to learn a baseline suspicious-transaction ranker. It loads every CSV and Parquet table, converts transactions into one effective sender-to-recipient payment, builds transaction and account-behaviour features, checks average precision on a held-out part of training data, and optionally writes scores for `test-data/`.

The provided Parquet tables are read directly with pandas/PyArrow; CSV conversion is not required. Keep `payments.py` beside this notebook. This baseline is a starting point, not a claim that any payment pattern proves fraud.

**Required packages:** `pandas`, `pyarrow`, `scikit-learn`, `numpy`.

## 1. Setup and load the full training folder

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path.cwd()
if not (DATA_DIR / "train").is_dir():
    project_dir = DATA_DIR / "XRPL_Decode_and_Detect_2026-3-1_Overfitting"
    if (project_dir / "train").is_dir():
        DATA_DIR = project_dir
    else:
        raise FileNotFoundError(f"Could not find train/ from {Path.cwd()}")

TRAIN_DIR = DATA_DIR / "train"
required_tables = {"transactions", "transfers", "blocks", "labels", "market_rates"}
train_data = {}
for data_file in sorted(TRAIN_DIR.iterdir()):
    if data_file.suffix.lower() == ".parquet":
        train_data[data_file.stem] = pd.read_parquet(data_file)
    elif data_file.suffix.lower() == ".csv":
        train_data[data_file.stem] = pd.read_csv(data_file)

missing_tables = required_tables - train_data.keys()
if missing_tables:
    raise FileNotFoundError(f"Missing required training tables in {TRAIN_DIR}: {sorted(missing_tables)}")

print(f"Loaded {len(train_data)} tables from {TRAIN_DIR}")
pd.DataFrame(
    {name: {"rows": len(table), "columns": len(table.columns)} for name, table in train_data.items()}
).T

## 2. Create one labelled row per transaction

The raw transfers table can contain multiple executed legs per payment. This baseline follows the project helper and uses one customer-level payment row per transaction, valued with the issuer-aware hourly reference rates. Labels are joined by transaction hash; annotation metadata is deliberately excluded from model features.

In [ ]:
import sys

if str(DATA_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_DIR))
from payments import project

rate_table = train_data["market_rates"]
train_rates = {
    (int(row.TIMESTAMP), row.ASSET): int(row.PRICE_MICRO_USD)
    for row in rate_table.itertuples(index=False)
}
projected = pd.DataFrame(
    project(train_data["transactions"].to_dict("records"), train_rates)
)
label_columns = train_data["labels"][ ["TRANSACTION_HASH", "is_suspicious"] ]
train = projected.merge(
    label_columns,
    on="TRANSACTION_HASH",
    how="inner",
    validate="one_to_one",
)
if len(train) != len(projected) or len(train) != len(label_columns):
    raise ValueError("Transaction and label hashes do not match one-to-one")

print(
    f"{len(train):,} payments; "
    f"{int(train['is_suspicious'].sum()):,} suspicious "
    f"({train['is_suspicious'].mean():.2%})"
)
train.head()

## 3. Build transaction and flow features

These features describe payment size, account activity and short-term pass-through behaviour. They use no labels. Because this is an offline, full-window dataset, account activity can use the other transactions in the same split; do not interpret the scores as real-time alerts.

In [ ]:
HOUR = 3600
TEN_MINUTES = 600


def count_in_window(times_by_account, account, timestamp, before, after):
    times = times_by_account.get(account)
    if times is None:
        return 0
    left = np.searchsorted(times, timestamp - before, side="left")
    right = np.searchsorted(times, timestamp + after, side="right")
    return int(right - left)


def make_features(payments):
    df = payments.sort_values("TIMESTAMP", kind="stable").reset_index(drop=True)
    timestamps = df["TIMESTAMP"].astype("int64").to_numpy()
    values = df["VALUE"].astype(float).to_numpy()
    sends = {
        account: group["TIMESTAMP"].astype("int64").to_numpy()
        for account, group in df.groupby("FROM_ADDRESS", sort=False)
    }
    receives = {
        account: group["TIMESTAMP"].astype("int64").to_numpy()
        for account, group in df.groupby("TO_ADDRESS", sort=False)
    }
    sent_values = {
        account: group["VALUE"].astype(float).to_numpy()
        for account, group in df.groupby("FROM_ADDRESS", sort=False)
    }
    received_values = {
        account: group["VALUE"].astype(float).to_numpy()
        for account, group in df.groupby("TO_ADDRESS", sort=False)
    }

    features = pd.DataFrame({"TRANSACTION_HASH": df["TRANSACTION_HASH"]})
    features["log_value"] = np.log1p(np.maximum(values, 0))
    features["sender_payment_count"] = df["FROM_ADDRESS"].map(df["FROM_ADDRESS"].value_counts())
    features["receiver_payment_count"] = df["TO_ADDRESS"].map(df["TO_ADDRESS"].value_counts())
    features["sender_burst_10m"] = [
        count_in_window(sends, account, timestamp, TEN_MINUTES, TEN_MINUTES)
        for account, timestamp in zip(df["FROM_ADDRESS"], timestamps)
    ]
    features["receiver_burst_10m"] = [
        count_in_window(receives, account, timestamp, TEN_MINUTES, TEN_MINUTES)
        for account, timestamp in zip(df["TO_ADDRESS"], timestamps)
    ]
    features["receiver_forwards_within_hour"] = [
        count_in_window(sends, account, timestamp, 0, HOUR) > 0
        for account, timestamp in zip(df["TO_ADDRESS"], timestamps)
    ]
    features["sender_received_within_hour"] = [
        count_in_window(receives, account, timestamp, HOUR, 0) > 0
        for account, timestamp in zip(df["FROM_ADDRESS"], timestamps)
    ]

    def has_similar_payment(account, timestamp, amount, times_by_account, values_by_account, *, after):
        times = times_by_account.get(account)
        if times is None:
            return False
        if after:
            start = np.searchsorted(times, timestamp, side="right")
            end = np.searchsorted(times, timestamp + HOUR, side="right")
        else:
            start = np.searchsorted(times, timestamp - HOUR, side="left")
            end = np.searchsorted(times, timestamp, side="left")
        nearby_values = values_by_account[account][start:end]
        return bool(np.any(np.abs(nearby_values - amount) <= 0.1 * max(amount, 1e-9)))

    features["receiver_forwards_similar_amount"] = [
        has_similar_payment(account, timestamp, amount, sends, sent_values, after=True)
        for account, timestamp, amount in zip(df["TO_ADDRESS"], timestamps, values)
    ]
    features["sender_received_similar_amount"] = [
        has_similar_payment(account, timestamp, amount, receives, received_values, after=False)
        for account, timestamp, amount in zip(df["FROM_ADDRESS"], timestamps, values)
    ]

    sent_totals = df.groupby("FROM_ADDRESS")["VALUE"].sum()
    received_totals = df.groupby("TO_ADDRESS")["VALUE"].sum()
    pass_through = (sent_totals / received_totals.reindex(sent_totals.index)).clip(0, 3)
    features["sender_pass_through"] = df["FROM_ADDRESS"].map(pass_through).fillna(0)
    features["receiver_pass_through"] = df["TO_ADDRESS"].map(pass_through).fillna(0)

    return features.fillna(0)


train_features = make_features(train)
feature_columns = [column for column in train_features.columns if column != "TRANSACTION_HASH"]
X = train_features[feature_columns].astype(float)
y = train.set_index("TRANSACTION_HASH")["is_suspicious"].loc[
    train_features["TRANSACTION_HASH"]
].to_numpy(dtype=int)
print(f"Built {X.shape[1]} features for {len(X):,} payments")
X.describe().T[["mean", "max"]]

## 4. Held-out training check

This random stratified split is a quick baseline check, not the competition test score. Average precision (AP) matches the leaderboard ranking metric; scores below are reported on the 0-100 scale.

In [ ]:
X_fit, X_check, y_fit, y_check = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
)
model.fit(X_fit, y_fit)
check_scores = model.predict_proba(X_check)[:, 1]
ap_score = average_precision_score(y_check, check_scores)
print(f"Held-out average precision: {100 * ap_score:.2f} / 100")

### Feature weights

The logistic-regression weights show which standardized features push predictions up or down. Treat them as model associations, not causal explanations.

In [ ]:
weights = pd.Series(model[-1].coef_[0], index=feature_columns).sort_values()
weights.round(3)

## 5. Optional: score the test package

If you have downloaded the separate package into `test-data/`, retrain on all labelled training payments and save one score per transaction as `predictions.csv`. The notebook skips this section when the test package is absent.

In [ ]:
TEST_DIR = DATA_DIR / "test-data"
if not TEST_DIR.is_dir() and (DATA_DIR / "test").is_dir():
    TEST_DIR = DATA_DIR / "test"

if TEST_DIR.is_dir():
    required_files = [TEST_DIR / "transactions.parquet", TEST_DIR / "market_rates.csv"]
    missing_files = [path.name for path in required_files if not path.is_file()]
    if missing_files:
        raise FileNotFoundError(f"Missing test files in {TEST_DIR}: {missing_files}")

    test_transactions = pd.read_parquet(required_files[0])
    test_rates_df = pd.read_csv(required_files[1])
    test_rates = {
        (int(row.TIMESTAMP), row.ASSET): int(row.PRICE_MICRO_USD)
        for row in test_rates_df.itertuples(index=False)
    }
    test_payments = pd.DataFrame(
        project(test_transactions.to_dict("records"), test_rates)
    )
    test_features = make_features(test_payments)
    if test_features["TRANSACTION_HASH"].duplicated().any():
        raise ValueError("Test feature rows contain duplicate transaction hashes")

    model.fit(X, y)
    test_scores = model.predict_proba(test_features[feature_columns])[:, 1]
    predictions = pd.DataFrame(
        {
            "TRANSACTION_HASH": test_features["TRANSACTION_HASH"],
            "score": test_scores,
        }
    )
    if predictions["score"].isna().any() or not predictions["score"].between(0, 1).all():
        raise ValueError("Predicted scores must be finite numbers between 0 and 1")

    output_path = DATA_DIR / "predictions.csv"
    predictions.to_csv(output_path, index=False)
    print(f"Saved {len(predictions):,} predictions to {output_path}")
    predictions.head()
else:
    print(f"No test package found at {TEST_DIR}; skipping prediction export.")

## Notes

- The payment projection values assets using the included hourly reference prices. These are valuations, not executable quotes.
- `labels.csv` contains scenario and cohort metadata; only `is_suspicious` is used as the target. Those metadata fields are never included as features.
- For a stronger validation estimate, use grouped/time-based splits and compare feature sets. The held-out result here is only a quick baseline.
- Keep `payments.py`, the training directory and the notebook together. Parquet is consumed directly; optional CSV conversion is unnecessary.